# Week 2 — Exploratory Data Analysis and Data Preprocessing
**DATS 511 · Applied Statistics and Data Analysis · Fall 2026**

This in-class notebook creates 554 fictional people directly in Python. Run the cells in order to follow each operation on the data.

[Open in Google Colab](https://colab.research.google.com/github/omzeybek/Yeditepe_Data_Science_DATS511/blob/main/Week2_EDA.ipynb)

Except for the dataset-generation cell, each cell has at most eight lines.

## 1. Import the tools
NumPy generates reproducible values; pandas stores and summarizes tables. Matplotlib and seaborn help display distributions and comparisons.

`Figure` lets us assemble a plot over several short cells and display the completed chart once. `display` shows tables and figures in the notebook.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.figure import Figure
from IPython.display import display

Set the random seed and consistent display preferences. Reusing the seed gives the same classroom data each time.

In [ ]:
SEED = 5112026
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

The versions below help reproduce the lesson. Colab usually provides these packages; local installation instructions are in the repository README.

In [ ]:
from importlib.metadata import version
packages = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn"]
package_versions = {name: version(name) for name in packages}
display(pd.Series(package_versions, name="installed_version"))

## 2. Prepare the synthetic data
All people and measurements are fictional and do not represent Türkiye's population. Wages are monthly TRY, wealth is TRY, height is cm, weight is kg, and home area is m². YKS values are simulated scores, not historical results.

The next cell defines the generator and is the one longer preparation cell. It creates plausible education/age relationships, a legitimate PhD example with 25 education years, and a dispersed positive wealth distribution.

The literal rates in the brief produce five missing fields and one injected measurement error: 1,000 education years. Missingness is deliberately placed in the five imputation variables. Natural IQR flags can be more numerous.

In [ ]:
"""Reproducible fictional Turkish-person data for a DATS 511 classroom exercise."""
import numpy as np
import pandas as pd

COLUMNS = [
    "name", "surname", "person_id", "age", "occupation", "sex", "weight_kg",
    "height_cm", "education_years", "education_level", "monthly_wage_try",
    "wealth_try", "home_sqm", "city", "yks_score", "university_completed",
    "department_completed",
]
NUMERIC_COLUMNS = [
    "age", "weight_kg", "height_cm", "education_years", "monthly_wage_try",
    "wealth_try", "home_sqm", "yks_score",
]
EDUCATION_ORDER = ["İlkokul", "Ortaokul", "Lise", "Ön Lisans", "Lisans", "Yüksek Lisans", "Doktora"]


def generate_dataset(missing_rate=0.0005, outlier_rate=0.0002, seed=5112026):
    """Return a 554-row DataFrame and a JSON-serializable provenance dictionary.

    Rates use all 17 columns for missingness and the 8 numeric columns for
    outliers. Thus 0.05% / 0.02% produce five missing cells / one error.
    Set 0.05 / 0.02 for the alternative 5% / 2% interpretation.
    Missingness is inserted only in numeric columns: the five variables
    requested for imputation are prioritized, then other numeric cells.
    """
    if not (0 <= missing_rate <= 1 and 0 <= outlier_rate <= 1):
        raise ValueError("Rates must be proportions between zero and one.")
    rng = np.random.default_rng(seed)
    n = 554
    sex = rng.choice(["Kadın", "Erkek"], n)
    female_names = ["Ayşe", "Fatma", "Zeynep", "Elif", "Merve", "Esra", "Derya", "Selin", "Ece", "İrem", "Özge", "Deniz"]
    male_names = ["Mehmet", "Mustafa", "Ahmet", "Ali", "Emre", "Burak", "Murat", "Can", "Kerem", "Oğuz", "Uğur", "Deniz"]
    names = [rng.choice(female_names if s == "Kadın" else male_names) for s in sex]
    surnames = rng.choice(["Yılmaz", "Kaya", "Demir", "Çelik", "Şahin", "Yıldız", "Öztürk", "Aydın", "Arslan", "Doğan", "Kılıç", "Koç", "Kurt", "Aksoy", "Güneş", "Özdemir"], n)
    level = rng.choice(EDUCATION_ORDER, n, p=[0.07, 0.08, 0.30, 0.13, 0.28, 0.10, 0.04])
    years_by_level = dict(zip(EDUCATION_ORDER, [5, 8, 12, 14, 16, 18, 23]))
    years = np.array([years_by_level[x] for x in level])
    doctorate = np.flatnonzero(level == "Doktora")
    years[doctorate] = rng.integers(22, 26, len(doctorate))
    level[0], years[0] = "Doktora", 25  # A valid long doctoral education.
    minimum_age = np.maximum(18, years + 6)
    age = np.maximum(rng.integers(18, 76, n), minimum_age)
    age[0] = 43
    jobs = {
        "İlkokul": ["İşçi", "Çiftçi", "Esnaf", "Ev işleriyle meşgul"],
        "Ortaokul": ["İşçi", "Şoför", "Esnaf", "Satış danışmanı"],
        "Lise": ["Satış danışmanı", "İdari personel", "Şoför", "İşletme sahibi"],
        "Ön Lisans": ["Tekniker", "Sağlık teknikeri", "İdari personel"],
        "Lisans": ["Mühendis", "Öğretmen", "Muhasebeci", "Yazılım geliştirici"],
        "Yüksek Lisans": ["Mühendis", "Yönetici", "Yazılım geliştirici", "Öğretmen"],
        "Doktora": ["Akademisyen", "Araştırmacı", "Yönetici"],
    }
    occupation = [rng.choice(jobs[x]) for x in level]
    for i in range(1, n):
        if age[i] >= 60 and rng.random() < 0.65:
            occupation[i] = "Emekli"
        elif age[i] < 23 and level[i] == "Lise" and rng.random() < 0.35:
            occupation[i] = "Öğrenci"
        elif rng.random() < 0.035:
            occupation[i] = "İşsiz"
    occupation[0] = "Akademisyen"
    height = np.clip(rng.normal(np.where(sex == "Kadın", 163, 176), 6.5), 143, 198)
    bmi = np.clip(rng.normal(25.5 + 0.018 * (age - 35), 3.2), 18.5, 36)
    weight = np.clip(bmi * (height / 100) ** 2, 43, 125)
    wage_base = dict(zip(EDUCATION_ORDER, [29000, 31000, 36000, 44000, 61000, 79000, 103000]))
    wage = np.array([wage_base[x] for x in level]) * rng.lognormal(0, 0.30, n)
    wage *= 0.85 + np.minimum(age - 18, 30) / 100
    for i, job in enumerate(occupation):
        if job == "Emekli":
            wage[i] *= 0.55
        elif job in ["Öğrenci", "İşsiz", "Ev işleriyle meşgul"]:
            wage[i] = 0
    # Strictly positive and strongly right-skewed: useful for a log-transform.
    wealth = rng.lognormal(np.log(2200000), 1.60, n) * (0.55 + age / 60)
    home = np.clip(95 + 20 * np.log1p(wealth / 2200000) + rng.normal(0, 25, n), 30, 430)
    cities = rng.choice(["İstanbul", "Ankara", "İzmir", "Bursa", "Antalya", "Adana", "Konya", "Gaziantep", "Eskişehir", "Samsun", "Trabzon", "Diyarbakır", "Kayseri", "Mersin"], n)
    universities = ["Yeditepe Üniversitesi", "İstanbul Üniversitesi", "Ankara Üniversitesi", "Ege Üniversitesi", "Hacettepe Üniversitesi", "Marmara Üniversitesi", "Dokuz Eylül Üniversitesi", "Orta Doğu Teknik Üniversitesi", "Anadolu Üniversitesi"]
    departments = {
        "Ön Lisans": ["Bilgisayar Programcılığı", "İşletme Yönetimi", "Tıbbi Laboratuvar Teknikleri"],
        "Lisans": ["Bilgisayar Mühendisliği", "İşletme", "İstatistik", "Matematik", "İktisat", "Sınıf Öğretmenliği"],
        "Yüksek Lisans": ["Veri Bilimi", "İşletme", "İstatistik", "Bilgisayar Mühendisliği", "İktisat"],
        "Doktora": ["İstatistik", "Bilgisayar Mühendisliği", "İktisat", "İşletme"],
    }
    # Job and qualification labels refer to broad education groups, not a
    # specific verified professional license or a person's real credentials.
    university = [rng.choice(universities) if x in departments else "Üniversite tamamlanmadı" for x in level]
    department = [rng.choice(departments[x]) if x in departments else "Bölüm tamamlanmadı" for x in level]
    yks = np.clip(230 + 13 * (years - 12) + rng.normal(0, 45, n), 150, 500)
    df = pd.DataFrame({
        "name": names, "surname": surnames,
        "person_id": [f"SYN{i + 1:04d}" for i in range(n)], "age": age,
        "occupation": occupation, "sex": sex, "weight_kg": np.round(weight, 1),
        "height_cm": np.round(height, 1), "education_years": years,
        "education_level": level, "monthly_wage_try": np.round(wage, 2),
        "wealth_try": np.round(wealth, 2), "home_sqm": np.round(home, 1),
        "city": cities, "yks_score": np.round(yks, 1),
        "university_completed": university, "department_completed": department,
    }, columns=COLUMNS)
    missing_target = int(round(n * len(COLUMNS) * missing_rate))
    outlier_target = int(round(n * len(NUMERIC_COLUMNS) * outlier_rate))
    injected = []
    protected = {(0, "education_years")}  # Retain the valid PhD example.
    if outlier_target:
        error_row = int(rng.integers(1, n))
        old = float(df.at[error_row, "education_years"])
        df.at[error_row, "education_years"] = 1000
        injected.append({"row_index": error_row, "person_id": df.at[error_row, "person_id"], "column": "education_years", "original_value": old, "injected_value": 1000, "kind": "measurement_error", "reason": "Impossible education duration; flag for explicit measurement-error handling."})
        protected.add((error_row, "education_years"))
    # At the higher optional rate, add plausible extreme financial values;
    # these are contextual outliers, not automatically measurement errors.
    eligible = [(i, col) for i in range(1, n) for col in ["wealth_try", "home_sqm", "monthly_wage_try"] if col != "monthly_wage_try" or occupation[i] not in ["Emekli", "Öğrenci", "İşsiz", "Ev işleriyle meşgul"]]
    if outlier_target - 1 > len(eligible):
        raise ValueError("Outlier rate exceeds available contextual-outlier cells.")
    if outlier_target > 1:
        for pos in rng.choice(len(eligible), size=outlier_target - 1, replace=False):
            row, col = eligible[int(pos)]
            old = float(df.at[row, col])
            limits = {"wealth_try": (120000000, 600000000), "monthly_wage_try": (450000, 850000), "home_sqm": (550, 1100)}
            new = round(float(rng.uniform(*limits[col])), 2 if col != "home_sqm" else 1)
            df.at[row, col] = new
            injected.append({"row_index": row, "person_id": df.at[row, "person_id"], "column": col, "original_value": old, "injected_value": new, "kind": "contextual_extreme", "reason": "Simulated high financial value, not an impossible measurement."})
            protected.add((row, col))
    missing = []
    priority = ["monthly_wage_try", "wealth_try", "height_cm", "weight_kg", "home_sqm"]
    for col in priority[:missing_target]:
        available = [i for i in range(1, n) if (i, col) not in protected and i not in [x[0] for x in missing]]
        row = int(rng.choice(available))
        missing.append((row, col))
    candidates = [(i, col) for i in range(n) for col in NUMERIC_COLUMNS if (i, col) not in protected and (i, col) not in missing]
    extra = missing_target - len(missing)
    if extra > len(candidates):
        raise ValueError("Missing rate exceeds available numeric cells.")
    if extra:
        missing.extend(candidates[int(p)] for p in rng.choice(len(candidates), extra, replace=False))
    missing_records = []
    for row, col in missing:
        missing_records.append({"row_index": row, "person_id": df.at[row, "person_id"], "column": col, "original_value": float(df.at[row, col])})
        df.at[row, col] = np.nan
    injected_pairs = {(x["row_index"], x["column"]) for x in injected}
    iqr_flags = {}
    for col in NUMERIC_COLUMNS:
        q1, q3 = df[col].quantile([0.25, 0.75])
        lower, upper = float(q1 - 1.5 * (q3 - q1)), float(q3 + 1.5 * (q3 - q1))
        rows = df.index[(df[col] < lower) | (df[col] > upper)].tolist()
        iqr_flags[col] = {"lower_fence": lower, "upper_fence": upper, "flagged_count": len(rows), "naturally_generated_rows": [int(i) for i in rows if (i, col) not in injected_pairs], "injected_rows": [int(i) for i in rows if (i, col) in injected_pairs]}
    metadata = {
        "synthetic": True, "disclosure": "All names, identifiers and measurements are fictional; this is a teaching sample, not representative population or current official economic data.",
        "seed": int(seed), "rows": n, "columns": COLUMNS, "numeric_columns": NUMERIC_COLUMNS,
        "education_order": EDUCATION_ORDER,
        "units": {"age": "years", "weight_kg": "kg", "height_cm": "cm", "education_years": "years", "monthly_wage_try": "TRY/month; zero for no wage", "wealth_try": "TRY; positive synthetic total wealth", "home_sqm": "m²", "yks_score": "simulated 150–500 YKS-scale score"},
        "yks_score_note": "Scores are fictional YKS-scale proxies for all adults, not historical exam results; older cohorts did not take YKS under its current name.",
        "missing_rate_requested": float(missing_rate), "missing_denominator": n * len(COLUMNS), "missing_count": len(missing), "missing_rate_actual": len(missing) / (n * len(COLUMNS)), "missing_locations": missing_records,
        "outlier_rate_requested": float(outlier_rate), "outlier_denominator": n * len(NUMERIC_COLUMNS), "injected_outlier_count": len(injected), "injected_rate_actual": len(injected) / (n * len(NUMERIC_COLUMNS)), "injected_outliers": injected,
        "outlier_note": "Requested outlier rate controls deliberate injections; natural values flagged by the IQR rule are recorded separately and may be valid.",
        "valid_phd_example": {"row_index": 0, "person_id": "SYN0001", "age": 43, "education_years": 25},
        "iqr_outliers": iqr_flags,
    }
    return df, metadata

### Create the DataFrame in Python
Call the generator and store its result directly in `df`. The separate `metadata` dictionary records the seed, units, and deliberate data-quality problems.

In [ ]:
df, metadata = generate_dataset(
    missing_rate=0.0005,
    outlier_rate=0.0002,
    seed=SEED,
)
display(df.head())

Check the sample size and the number of missing fields before analysis. The raw DataFrame will remain available when we create a cleaned copy later.

In [ ]:
print("Rows and columns:", df.shape)
print("Missing fields:", int(df.isna().sum().sum()))
print("Impossible schooling values:", int(df["education_years"].eq(1000).sum()))
assert df.shape == (554, 17)
assert df["person_id"].is_unique

### Name the numeric features and the education order
Names and identifiers are labels. Education level is ordered; its category labels are not equally spaced numeric measurements.

In [ ]:
numeric_cols = NUMERIC_COLUMNS.copy()
education_order = EDUCATION_ORDER.copy()
display(pd.Series(numeric_cols, name="numeric_features"))

Use explicit units and readable titles throughout the plots.

In [ ]:
units = dict(zip(numeric_cols, [
    "years", "kg", "cm", "years",
    "TRY/month", "TRY", "m²", "points",
]))
titles = dict(zip(numeric_cols, [
    "Age", "Weight", "Height", "Years of education",
    "Monthly wage", "Wealth", "Home area", "Fictional YKS score",
]))

## 3. Explore the raw data

These analyses use the in-memory `df`, including five missing fields and the deliberately impossible **1,000 years of education**. No rows are removed or imputed here. The following cells show each operation separately: select measurements → summarize → calculate plotting inputs → draw → label → display.

Age and education are measured in years; weight in kg; height in cm; wage in TRY/month; wealth in TRY; home area in m²; and fictional YKS scores in points. TRY amounts have no implied valuation date or purchasing-power adjustment. YKS values are exercise variables, not historical examination results.

The results describe generated observations. They are neither population estimates for Türkiye nor evidence of causes. Names and identifiers are labels, not quantitative measurements.

**Select numeric measurements and plotting tools.** Converting a view leaves `df` unchanged.

In [ ]:
from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter, MaxNLocator
raw_numeric = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
categorical_cols = [column for column in df if column not in numeric_cols]
number_formatter = FuncFormatter(lambda value, position: f"{value:,.0f}")
MISSING_LABEL = "(Missing)"

**Inspect the schema.** Each missing percentage uses all rows as its denominator.

In [ ]:
schema = pd.DataFrame({"dtype": df.dtypes.astype(str)})
schema["non_missing"] = df.notna().sum()
schema["missing"] = df.isna().sum()
schema["distinct_non_missing"] = df.nunique(dropna=True)
schema["missing_percent"] = 100 * schema["missing"] / len(df)
print(f"Raw shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(schema)

**Read pandas' overview.** Categorical `top` and `freq` describe labels; numeric summaries describe quantities.

In [ ]:
df.info()
display(df.describe(include="all").T)

### Numeric summaries and IQR screening

Quartiles split the observed values into four parts. The interquartile range is **IQR = Q3 − Q1**. Tukey fences flag values below **Q1 − 1.5 × IQR** or above **Q3 + 1.5 × IQR**. These statistical flags do not prove measurement errors. Missing values are excluded separately for each numeric variable.

**Calculate quartiles and fences for every numeric column.** pandas aligns each threshold with its column name.

In [ ]:
q1 = raw_numeric.quantile(0.25)
q3 = raw_numeric.quantile(0.75)
iqr = q3 - q1
lower_fences = q1 - 1.5 * iqr
upper_fences = q3 + 1.5 * iqr
outlier_flags = raw_numeric.lt(lower_fences) | raw_numeric.gt(upper_fences)

**Attach outlier counts and measurement units to the descriptive table.**

In [ ]:
raw_numeric_summary = raw_numeric.describe().T
raw_numeric_summary = raw_numeric_summary.rename(columns={"count": "n", "50%": "median"})
raw_numeric_summary["IQR"] = iqr
raw_numeric_summary["Tukey_outliers"] = outlier_flags.sum()
raw_numeric_summary["outlier_percent_of_observed"] = 100 * outlier_flags.sum() / raw_numeric.count()
raw_numeric_summary["unit"] = pd.Series(units)
display(raw_numeric_summary.round(2))

**Compare means and medians.** Differences can reflect skewness or influential extremes. The 1,000-year education error still affects these raw statistics.

In [ ]:
insight_columns = ["age", "monthly_wage_try", "wealth_try", "education_years"]
insight_fields = ["n", "mean", "median", "Tukey_outliers", "unit"]
display(raw_numeric_summary.loc[insight_columns, insight_fields].round(2))

### Frequencies for nominal and ordinal variables

Names, IDs, sex, occupation, city, university, and department are nominal labels. Education level is ordinal: its categories follow the declared education order, but distances between category codes have no quantitative meaning.

Every frequency denominator is **all rows**, including missing labels. High-cardinality columns show the ten most frequent labels plus a combined remainder. Identifiers and names are inspected, not plotted.

**Make missing categories explicit, then count labels.** This short helper is reused wherever we group categories.

In [ ]:
def as_labels(series):
    return series.astype("string").fillna(MISSING_LABEL)

frequency_counts = {}
for column in categorical_cols:
    frequency_counts[column] = as_labels(df[column]).value_counts()

**Keep the ordered education categories and summarize long label lists.** The remainder preserves the total count.

In [ ]:
for column, counts in frequency_counts.items():
    if column == "education_level":
        extra = [label for label in counts.index if label not in education_order]
        frequency_counts[column] = counts.reindex(education_order + extra, fill_value=0)
    elif len(counts) > 10:
        short_counts = counts.iloc[:10].copy()
        short_counts.loc[f"(Other {len(counts) - 10} labels combined)"] = counts.iloc[10:].sum()
        frequency_counts[column] = short_counts

**Convert counts to relative frequencies and percentages.** Each table still sums to the full sample.

In [ ]:
for column, counts in frequency_counts.items():
    table = counts.rename("count").to_frame()
    table["relative_frequency"] = table["count"] / len(df)
    table["percent_of_all_rows"] = 100 * table["relative_frequency"]
    print(f"{column}: {df[column].nunique(dropna=True)} observed labels; n = {len(df):,}")
    display(table.round(4))

### Annotated boxplots for all eight numeric variables

Each panel has its own scale and units. The blue median line, green mean diamond, and purple mode crosses summarize location. At most three tied modes are shown. A mode may reflect rounding; when every observation is unique there is no informative empirical mode.

All IQR-flagged points are visible, with their lowest and highest values labeled. The annotation reports the total flagged count, including unlabelled points. The impossible education value remains visible and compresses that panel's central distribution.

We build a standalone `Figure` across several cells and call `display(fig)` only when it is complete. This avoids displaying an unfinished plot after every small step.

**Collect observed values, frequencies, and repeated modes.** These dictionaries have one entry per numeric variable.

In [ ]:
observed_values = {column: raw_numeric[column].dropna() for column in numeric_cols}
value_counts = {column: values.value_counts() for column, values in observed_values.items()}
repeated_modes = {}
for column, counts in value_counts.items():
    repeated = counts[counts.eq(counts.max()) & counts.gt(1)]
    repeated_modes[column] = repeated.index.sort_values().to_numpy()

**Create readable mode labels.** Extra tied modes remain counted even when only three are drawn.

In [ ]:
mode_labels = {}
for column, modes in repeated_modes.items():
    mode_labels[column] = ", ".join(f"{value:,.2f}" for value in modes[:3])
    if not len(modes):
        mode_labels[column] = "no repeated value"
    if len(modes) > 3:
        mode_labels[column] += f" (+{len(modes) - 3} tied)"

**Start the eight-panel figure and draw the boxes.** Outlier points are added separately so every flag remains visible.

In [ ]:
fig = Figure(figsize=(15, 19), layout="constrained")
axes = fig.subplots(4, 2)
for ax, column in zip(axes.flat, numeric_cols):
    ax.boxplot(observed_values[column], positions=[1], widths=0.45,
               showfliers=False, medianprops={"color": "royalblue", "linewidth": 2})
    ax.set_title(f"{titles[column]} — {units[column]}")

**Overlay means and repeated modes.** Their markers are intentionally different from the boxplot median.

In [ ]:
for ax, column in zip(axes.flat, numeric_cols):
    values = observed_values[column]
    modes = repeated_modes[column][:3]
    ax.scatter([1], [values.mean()], marker="D", color="seagreen", s=48, zorder=4)
    ax.scatter(np.ones(len(modes)), modes, marker="x", color="purple", s=65, zorder=4)

**Overlay all observations beyond the fences.** Deterministic horizontal spacing makes overlapping points easier to see.

In [ ]:
flagged_values = {}
for ax, column in zip(axes.flat, numeric_cols):
    flagged = raw_numeric.loc[outlier_flags[column], column].sort_values()
    flagged_values[column] = flagged
    offsets = np.linspace(-0.10, 0.10, len(flagged)) if len(flagged) > 1 else np.zeros(len(flagged))
    ax.scatter(1 + offsets, flagged, s=24, alpha=0.7, color="crimson", zorder=3)

**Label the two most extreme flagged values.** Every flagged point is drawn, but limiting text keeps each panel readable.

In [ ]:
for ax, column in zip(axes.flat, numeric_cols):
    extremes = flagged_values[column].iloc[[0, -1]].unique() if len(flagged_values[column]) else []
    for position, value in enumerate(extremes):
        ax.annotate(f"{value:,.2f}", xy=(1, value),
                    xytext=(35, 10 if position else -10), textcoords="offset points",
                    fontsize=8, color="crimson",
                    arrowprops={"arrowstyle": "-", "color": "crimson"})

**Assemble the numeric annotations from the same summary table.** Counts use each variable's observed denominator.

In [ ]:
box_notes = {}
for column, row in raw_numeric_summary.iterrows():
    box_notes[column] = (
        f"n = {row['n']:,.0f} | mean = {row['mean']:,.2f}\n"
        f"median = {row['median']:,.2f}\nmode = {mode_labels[column]}\n"
        f"IQR = {row['IQR']:,.2f}\nTukey outliers = {row['Tukey_outliers']:,.0f} "
        f"({row['outlier_percent_of_observed']:.1f}%)"
    )

**Add annotations and units.** Numbers use thousands separators rather than scientific offsets.

In [ ]:
for ax, column in zip(axes.flat, numeric_cols):
    ax.text(0.02, 0.98, box_notes[column], transform=ax.transAxes, va="top", fontsize=8,
            bbox={"facecolor": "white", "alpha": 0.9, "edgecolor": "#dddddd"})
    ax.set_xticks([])
    ax.set_xlim(0.5, 1.75)
    ax.set_ylabel(units[column])
    ax.yaxis.set_major_formatter(number_formatter)
    ax.grid(axis="y", alpha=0.2)

**Display the completed boxplots.** Compare each mean with its median and relate the red points to the IQR flags.

In [ ]:
fig.suptitle("Raw numeric boxplots: all observed values retained", fontsize=15)
display(fig)

### Histograms for all eight numeric variables

Histogram positions represent measured quantities. Dashed green lines mark means and dotted orange lines mark medians. We use unit-width bins for short integer ranges; otherwise we cap a Freedman–Diaconis bin suggestion at 35 bins. Bin choices affect apparent shape. Every observed value, including the education error, stays within the displayed range.

**Define a small bin-selection operation.** The generated data have observed values in every numeric column.

In [ ]:
def practical_histogram_bins(values, cap=35):
    data = np.asarray(values, dtype=float)
    if np.all(np.isclose(data, np.round(data))) and np.ptp(data) <= cap:
        return np.arange(data.min() - 0.5, data.max() + 1.5, 1)
    suggested = len(np.histogram_bin_edges(data, bins="fd")) - 1
    return min(cap, max(10, suggested))

**Draw the observed distributions.** Every panel includes its own number of observed values.

In [ ]:
fig = Figure(figsize=(15, 17), layout="constrained")
axes = fig.subplots(4, 2)
for ax, column in zip(axes.flat, numeric_cols):
    values = observed_values[column]
    ax.hist(values, bins=practical_histogram_bins(values),
            color="#397fa8", edgecolor="white", alpha=0.85)
    ax.set_title(f"{titles[column]} (n = {len(values):,})")

**Mark means and medians, then label each measured axis.**

In [ ]:
for ax, column in zip(axes.flat, numeric_cols):
    values = observed_values[column]
    ax.axvline(values.mean(), color="seagreen", linestyle="--", label="Mean")
    ax.axvline(values.median(), color="darkorange", linestyle=":", label="Median")
    ax.set_xlabel(units[column])
    ax.set_ylabel("Observed count")
    ax.legend(fontsize=8)

**Format count axes and display.** A long upper tail can separate the mean from the median.

In [ ]:
for ax in axes.flat:
    ax.xaxis.set_major_formatter(number_formatter)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(axis="y", alpha=0.2)
fig.suptitle("Raw numeric histograms: all observed values retained", fontsize=15)
display(fig)

### Monthly wage by ordered education level

The boxplot compares wage distributions, while the stacked histogram counts how each education group contributes to common wage intervals. Larger groups contribute more histogram counts. Missing education labels, if present, are retained as groups; missing wages are omitted and counted below. Differences reflect this fictional sample and do not estimate the wage effect of obtaining a degree.

**Select wages within each ordered education group.** The same group arrays feed both wage plots.

In [ ]:
education_labels = as_labels(df["education_level"])
extra_education = [label for label in education_labels.unique() if label not in education_order]
education_groups = education_order + extra_education
wage_groups = {label: raw_numeric.loc[education_labels.eq(label), "monthly_wage_try"].dropna()
               for label in education_groups}
nonempty_education = {label: values for label, values in wage_groups.items() if len(values)}
education_colors = sns.color_palette("viridis", len(nonempty_education))

**Create the grouped boxplot.** Labels include each group's observed sample size; green diamonds mark means.

In [ ]:
fig = Figure(figsize=(12, 6), layout="constrained")
ax = fig.subplots()
wage_arrays = list(nonempty_education.values())
wage_labels = [f"{label}\n(n={len(values)})" for label, values in nonempty_education.items()]
_ = ax.boxplot(wage_arrays, tick_labels=wage_labels, showmeans=True,
           meanprops={"marker": "D", "markerfacecolor": "seagreen", "markeredgecolor": "seagreen"})

**Label and display the wage comparisons.** Vertical positions remain measured in TRY per month.

In [ ]:
ax.set_title("Monthly wage by education level — raw data")
ax.set_ylabel("Monthly wage (TRY/month)")
ax.yaxis.set_major_formatter(number_formatter)
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.2)
display(fig)

**Build the stacked wage histogram with common bins.** Shared boundaries make group counts comparable within each interval.

In [ ]:
all_wages = observed_values["monthly_wage_try"]
fig = Figure(figsize=(12, 6), layout="constrained")
ax = fig.subplots()
ax.hist(wage_arrays, bins=practical_histogram_bins(all_wages, cap=25), stacked=True,
        label=list(nonempty_education), color=education_colors, edgecolor="white", alpha=0.85)
_ = ax.legend(title="Education level", fontsize=8)

**Display the histogram and report omissions.** Counts use only rows with observed wages.

In [ ]:
ax.set_title("Monthly wage histogram stacked by education level")
ax.set_xlabel("Monthly wage (TRY/month)")
ax.set_ylabel("Observed count")
ax.xaxis.set_major_formatter(number_formatter)
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(axis="y", alpha=0.2)
display(fig)
print(f"Both wage plots omit {raw_numeric['monthly_wage_try'].isna().sum()} missing wages.")

### Empirical category probabilities

Education and occupation bars show **count ÷ all rows**, including missing labels. They are category bars, not histograms of numeric category codes. The probabilities sum to one for each variable, but describe only this synthetic sample.

**Count the full category sets.** Education retains its ordinal order; occupations follow descending frequency.

In [ ]:
occupation_labels = as_labels(df["occupation"])
occupation_order = occupation_labels.value_counts().index.tolist()
category_probabilities = {}
category_probabilities["Education level"] = education_labels.value_counts().reindex(education_groups, fill_value=0) / len(df)
category_probabilities["Occupation"] = occupation_labels.value_counts().reindex(occupation_order) / len(df)
fig = Figure(figsize=(16, 7), layout="constrained")
axes = fig.subplots(1, 2)

**Draw bars and put a probability label on each bar.** A category's position is a label, not a numeric measurement.

In [ ]:
for ax, (title, probabilities) in zip(axes, category_probabilities.items()):
    bars = ax.barh(probabilities.index, probabilities, color="#447e9e")
    ax.bar_label(bars, labels=[f"{probability:.1%}" for probability in probabilities], padding=3)
    ax.invert_yaxis()
    ax.set_xlim(0, max(0.12, probabilities.max() * 1.25))
    ax.set_title(title)

**Display the probabilities.** All rows remain in the denominator, even when a category is missing.

In [ ]:
for ax in axes:
    ax.set_xlabel("Empirical probability (count ÷ all rows)")
    ax.grid(axis="x", alpha=0.2)
display(fig)

### Age stem-and-leaf display

The key **2 | 3 = 23 years** separates each age into tens and remaining years. Repeated leaves preserve repeated observations. Fractional leaves would be retained rather than rounded away.

**Sort ages and split them into stems and leaves.** Missing ages cannot contribute a leaf.

In [ ]:
ages = observed_values["age"].sort_values()
stems = np.floor(ages / 10).astype(int)
print("Age stem-and-leaf | key: 2 | 3 = 23 years")
for stem in range(int(stems.min()), int(stems.max()) + 1):
    leaves = ages.loc[stems.eq(stem)] - 10 * stem
    print(f"{stem:>3} | " + " ".join(f"{leaf:g}" for leaf in leaves))
print(f"Includes {len(ages):,} observed ages; {raw_numeric['age'].isna().sum()} missing ages omitted.")

### Mirrored sex counts by occupation and completed degree

Women (**Kadın**) appear on the left and men (**Erkek**) on the right. Negative coordinates only position bars; tick labels show absolute counts. These pyramids describe sample composition, not sex-specific rates or an age structure. “Degree” here means ordered `education_level`, including pre-university education; department is a field of study.

**Select the two displayed sex labels and report exclusions.** Missing or other sex labels are counted separately.

In [ ]:
sex_labels = as_labels(df["sex"])
sex_order = ["Kadın", "Erkek"]
other_sex = ~sex_labels.isin(sex_order)
sex_palette = {"Kadın": "#a04e89", "Erkek": "#447e9e"}
print(f"Two-sided charts include {(~other_sex).sum():,} rows; omit {other_sex.sum()} other/missing sex labels.")
if other_sex.any():
    display(sex_labels.loc[other_sex].value_counts().rename("count").to_frame())

**Cross-tabulate categories against sex.** Reindexing retains empty groups in the intended order.

In [ ]:
occupation_sex_counts = pd.crosstab(occupation_labels, sex_labels)
occupation_sex_counts = occupation_sex_counts.reindex(index=occupation_order, columns=sex_order, fill_value=0)
education_sex_counts = pd.crosstab(education_labels, sex_labels)
education_sex_counts = education_sex_counts.reindex(index=education_groups, columns=sex_order, fill_value=0)
sex_tables = {"Occupation": occupation_sex_counts, "Completed education": education_sex_counts}
fig = Figure(figsize=(17, max(6, len(occupation_order) * 0.36)), layout="constrained")
axes = fig.subplots(1, 2)

**Mirror the counts around zero.** A shared limit within each panel keeps the two sexes visually comparable.

In [ ]:
for ax, (title, table) in zip(axes, sex_tables.items()):
    ax.barh(table.index, -table["Kadın"], color=sex_palette["Kadın"], label="Kadın (left)")
    ax.barh(table.index, table["Erkek"], color=sex_palette["Erkek"], label="Erkek (right)")
    limit = max(1, table.to_numpy().max()) * 1.2
    ax.set_xlim(-limit, limit)
    ax.invert_yaxis()
    ax.set_title(title)

**Format absolute counts and display.** Group sizes explain the bar lengths; these are not normalized rates.

In [ ]:
for ax in axes:
    ax.axvline(0, color="#555555", linewidth=0.8)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=7))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{abs(value):,.0f}"))
    ax.set_xlabel("Observed people (absolute counts)")
    ax.grid(axis="x", alpha=0.2)
    ax.legend(fontsize=8)
display(fig)

### Home area, occupation, and sex

The stacked histogram places **home area in m²** on a measured axis and stacks occupation counts within intervals. Unequal interval widths mean heights are **counts per interval**, not comparable densities per m². If there are more than seven occupations, retain the six largest and combine all others explicitly.

The following pyramids compare women and men within each occupation and home-size interval. Exhaustive intervals retain negative values as quality flags and large homes as a 200+ group. Missing home area and sex labels outside the two displayed sides are omitted and counted below. These associations do not show that occupation causes home size.

**Define readable occupation groups for the home-area charts.** Every occupation remains represented.

In [ ]:
home = raw_numeric["home_sqm"]
observed_home = home.dropna()
main_occupations = occupation_order[:6] if len(occupation_order) > 7 else occupation_order
home_occupation = occupation_labels.where(occupation_labels.isin(main_occupations), "Other occupations (combined)")
home_occupation_order = main_occupations.copy()
if len(occupation_order) > 7:
    home_occupation_order.append("Other occupations (combined)")
home_colors = sns.color_palette("tab10", len(home_occupation_order))

**Choose histogram boundaries that cover all observed homes.** Fixed middle boundaries support interpretable home-size intervals.

In [ ]:
lower_edge = min(0.0, np.floor(observed_home.min() / 50) * 50)
upper_edge = max(250.0, (np.floor(observed_home.max() / 50) + 1) * 50)
home_histogram_edges = np.unique([lower_edge, 0, 50, 75, 100, 150, 200, upper_edge])
home_arrays = [home.loc[home_occupation.eq(label)].dropna() for label in home_occupation_order]
fig = Figure(figsize=(13, 6), layout="constrained")
ax = fig.subplots()
_ = ax.hist(home_arrays, bins=home_histogram_edges, stacked=True, color=home_colors,
        label=home_occupation_order, edgecolor="white", alpha=0.85)

**Label and display the stacked home-area histogram.** Unequal-width bins deliberately retain their measured positions.

In [ ]:
ax.set_title("Home-area histogram stacked by occupation (unequal-width intervals)")
ax.set_xlabel("Home area (m²)")
ax.set_ylabel("Observed count per interval")
ax.set_xticks(home_histogram_edges)
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(title="Occupation", fontsize=8)
ax.grid(axis="y", alpha=0.2)
display(fig)

**Assign every observed home to an exhaustive size interval.** The left boundary is included and the right boundary excluded.

In [ ]:
home_bin_labels = ["Below 0 (check)", "0–<50", "50–<75", "75–<100", "100–<150", "150–<200", "200+"]
home_bin_edges = [-np.inf, 0, 50, 75, 100, 150, 200, np.inf]
home_bins = pd.cut(home, bins=home_bin_edges, labels=home_bin_labels, right=False)
home_sex_tables = {}
for occupation in home_occupation_order:
    selection = home_occupation.eq(occupation) & home.notna()
    counts = pd.crosstab(home_bins.loc[selection], sex_labels.loc[selection], dropna=False)
    home_sex_tables[occupation] = counts.reindex(index=home_bin_labels, columns=sex_order, fill_value=0)

**Create one pyramid panel per occupation group.** Two columns provide space for the bin labels.

In [ ]:
n_home_panels = len(home_occupation_order)
n_home_rows = (n_home_panels + 1) // 2
fig = Figure(figsize=(15, 4 * n_home_rows), layout="constrained")
axes = fig.subplots(n_home_rows, 2, squeeze=False)
for ax, (occupation, counts) in zip(axes.flat, home_sex_tables.items()):
    ax.barh(counts.index, -counts["Kadın"], color=sex_palette["Kadın"], label="Kadın (left)")
    ax.barh(counts.index, counts["Erkek"], color=sex_palette["Erkek"], label="Erkek (right)")
    ax.set_title(f"{occupation} (shown n = {counts.to_numpy().sum()})")

**Use symmetric count axes and keep home-size categories ordered.** Each panel has its own count scale.

In [ ]:
for ax, counts in zip(axes.flat, home_sex_tables.values()):
    limit = max(1, counts.to_numpy().max()) * 1.2
    ax.set_xlim(-limit, limit)
    ax.invert_yaxis()
    ax.axvline(0, color="#555555", linewidth=0.8)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=7))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{abs(value):,.0f}"))
    ax.set_xlabel("Observed people (absolute counts); home bins in m²")

**Complete the pyramid layout and display.** Unused panels are hidden when the group count is odd.

In [ ]:
for ax in list(axes.flat)[:n_home_panels]:
    ax.grid(axis="x", alpha=0.2)
    ax.legend(fontsize=8)
for ax in list(axes.flat)[n_home_panels:]:
    ax.set_visible(False)
fig.suptitle("Home-area and sex pyramids within occupation groups", fontsize=15)
display(fig)

**Audit omissions and the underlying home counts.** The histogram includes all sex labels; the two-sided pyramids include only Kadın/Erkek.

In [ ]:
print(f"Stacked histogram omits {home.isna().sum()} missing home areas.")
print(f"Pyramids omit {(home.isna() | other_sex).sum()} rows missing home area or a displayed sex label.")
home_occupation_counts = pd.crosstab(home_bins, home_occupation, dropna=False)
display(home_occupation_counts.reindex(index=home_bin_labels, columns=home_occupation_order, fill_value=0))

### Monthly wage versus wealth

Each scatterplot point represents a row with both values observed. Sex is a category color, never a numeric code. Pearson correlation measures linear association and is sensitive to extreme values; Spearman correlation measures rank association. Neither establishes causation, and their patterns partly reflect the simulation design.

**Select complete wage–wealth pairs and draw the category-colored points.** Missing pairs are excluded only from this analysis.

In [ ]:
paired = raw_numeric[["monthly_wage_try", "wealth_try"]].dropna()
fig = Figure(figsize=(11, 7), layout="constrained")
ax = fig.subplots()
for label in sex_labels.loc[paired.index].unique():
    group = paired.loc[sex_labels.loc[paired.index].eq(label)]
    ax.scatter(group["monthly_wage_try"], group["wealth_try"], s=24, alpha=0.55,
               color=sex_palette.get(label, "#777777"), label=label)

**Label measured axes and display.** Both quantities have different units despite sharing the same currency.

In [ ]:
ax.set_title(f"Monthly wage versus wealth — {len(paired):,} complete pairs")
ax.set_xlabel("Monthly wage (TRY/month)")
ax.set_ylabel("Wealth (TRY)")
ax.xaxis.set_major_formatter(number_formatter)
ax.yaxis.set_major_formatter(number_formatter)
ax.grid(alpha=0.2)
ax.legend(title="Sex", fontsize=9)
display(fig)

**Compare linear and rank associations.** Read them alongside the scatterplot rather than as causal conclusions.

In [ ]:
print(f"Pairwise analysis omits {len(df) - len(paired):,} rows missing either value.")
print(f"Pearson r: {paired.corr(method='pearson').iloc[0, 1]:.3f}")
print(f"Spearman rho: {paired.corr(method='spearman').iloc[0, 1]:.3f}")
print("These are descriptive associations in synthetic data, not causal effects.")

## 4. Data quality and preprocessing

Keep the Python-generated data in `df_raw`; apply changes to a separate `df_clean`.
Inspect missingness, dominant values and IQR flags before choosing any treatment.
An IQR flag invites review: high wealth and 25 years of postgraduate education can be valid.
For future predictive modelling, split training/test data before fitting any preprocessing.

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

df_raw = df.copy(deep=True)
assert len(df_raw) == 554

### Count missing entries in every column

`isna()` produces a Boolean table; summing each column counts its missing entries.
The percentage denominator is all people, including those with incomplete records.

In [ ]:
missing_analysis = pd.DataFrame(index=df_raw.columns.copy())
missing_analysis["dtype"] = df_raw.dtypes.astype(str)
missing_analysis["nonmissing_count"] = df_raw.notna().sum()
missing_analysis["missing_count"] = df_raw.isna().sum()
missing_analysis["missing_pct"] = (100 * df_raw.isna().mean()).round(2)
missing_analysis.index.name = "column"
display(missing_analysis)

### Find columns dominated by one value

Count the most common observed value first; treat missingness as a separate bucket.
Flag a column only when a bucket occupies **strictly more than 75% of all rows**.
This is a review flag, not an instruction to delete a column.

In [ ]:
observed_modes = {}
observed_mode_counts = {}
for column in df_raw.columns:
    counts = df_raw[column].value_counts(dropna=True)
    observed_modes[column] = counts.index[0] if not counts.empty else "<none>"
    observed_mode_counts[column] = int(counts.iloc[0]) if not counts.empty else 0

In [ ]:
concentration_analysis = pd.DataFrame(index=df_raw.columns.copy())
concentration_analysis["distinct_observed_values"] = df_raw.nunique()
concentration_analysis["most_common_observed_value"] = pd.Series(observed_modes)
concentration_analysis["most_common_observed_count"] = pd.Series(observed_mode_counts)
observed_count = concentration_analysis["most_common_observed_count"]
concentration_analysis["most_common_observed_pct_of_all_rows"] = 100 * observed_count / len(df_raw)
concentration_analysis["missing_pct_of_all_rows"] = 100 * df_raw.isna().sum() / len(df_raw)
concentration_analysis.index.name = "column"

In [ ]:
missing_dominates = df_raw.isna().sum().gt(observed_count)
dominant_value = concentration_analysis["most_common_observed_value"].copy()
dominant_value = dominant_value.mask(missing_dominates, "<Missing>")
dominant_count = observed_count.where(~missing_dominates, df_raw.isna().sum())
concentration_analysis["dominant_bucket_including_missing"] = dominant_value
concentration_analysis["dominant_bucket_count"] = dominant_count
concentration_analysis["dominant_bucket_pct"] = 100 * dominant_count / len(df_raw)
concentration_analysis["more_than_75pct_identical"] = dominant_count / len(df_raw) > 0.75

In [ ]:
display(concentration_analysis.round(2))
dominant_mask = concentration_analysis["more_than_75pct_identical"]
dominant_columns = concentration_analysis.index[dominant_mask].tolist()
print("Columns exceeding 75% concentration:", dominant_columns or "None")

### Calculate the IQR screening limits

For each numeric column, calculate \(IQR = Q_3 - Q_1\).
Values below \(Q_1 - 1.5\,IQR\) or above \(Q_3 + 1.5\,IQR\) receive a review flag.
Quartiles ignore missing entries; a statistical flag alone does not prove a measurement error.

In [ ]:
raw_numeric = df_raw[numeric_cols]
raw_iqr_summary = pd.DataFrame(index=numeric_cols)
raw_iqr_summary["observed_count"] = raw_numeric.notna().sum()
raw_iqr_summary["q1"] = raw_numeric.quantile(0.25)
raw_iqr_summary["q3"] = raw_numeric.quantile(0.75)
raw_iqr_summary["iqr"] = raw_iqr_summary["q3"] - raw_iqr_summary["q1"]
raw_iqr_summary["lower_fence"] = raw_iqr_summary["q1"] - 1.5 * raw_iqr_summary["iqr"]
raw_iqr_summary["upper_fence"] = raw_iqr_summary["q3"] + 1.5 * raw_iqr_summary["iqr"]

In [ ]:
raw_below = raw_numeric.lt(raw_iqr_summary["lower_fence"], axis="columns")
raw_above = raw_numeric.gt(raw_iqr_summary["upper_fence"], axis="columns")
raw_iqr_flags = raw_below | raw_above
raw_iqr_summary["below_fence_count"] = raw_below.sum()
raw_iqr_summary["above_fence_count"] = raw_above.sum()
raw_iqr_summary["iqr_flag_count"] = raw_iqr_flags.sum()
display(raw_iqr_summary.round(2))
print("Records with an IQR flag:", int(raw_iqr_flags.any(axis=1).sum()))

### Investigate education values in context

Schooling of 1,000 literal years is impossible; 25 years can describe advanced postgraduate study.
Review schooling above a person's age, but do not use that check alone to delete a record.
The chosen error rule is negative schooling or schooling above 100 years.

In [ ]:
schooling = df_raw["education_years"]
schooling_context_flags = pd.DataFrame(index=df_raw.index)
schooling_context_flags["negative_schooling"] = schooling.lt(0)
schooling_context_flags["schooling_above_100_years"] = schooling.gt(100)
schooling_context_flags["schooling_exceeds_age_review"] = schooling.gt(df_raw["age"])
schooling_context_flags["unusual_but_potentially_plausible_25_years"] = schooling.eq(25)
measurement_error_mask = schooling.lt(0) | schooling.gt(100)
measurement_error_rows = df_raw.loc[measurement_error_mask].copy()

In [ ]:
review_columns = ["person_id", "age", "education_years", "education_level",
                  "university_completed", "department_completed"]
review_mask = schooling_context_flags.any(axis=1)
context_review = df_raw.loc[review_mask, review_columns]
context_review = context_review.join(schooling_context_flags.loc[review_mask])
display(context_review)
assert len(measurement_error_rows) == 1
assert measurement_error_rows["education_years"].eq(1000).all()

Exclude the one impossible record **before fitting imputers or scaling distances**.
There is no trusted replacement in this exercise; the original record remains available for review.
With a verified source, correcting the single value could preserve the rest of that record.
Keep plausible long education and all other valid distribution tails.

In [ ]:
df_clean = df_raw.loc[~measurement_error_mask].copy(deep=True)
assert len(df_clean) == len(df_raw) - len(measurement_error_rows)
assert df_clean["education_years"].eq(25).sum() == df_raw["education_years"].eq(25).sum()
print(f"Retained {len(df_clean)} people; no record was removed for an IQR flag alone.")

### Record the values that must be preserved

Take an in-memory snapshot of observed values before filling gaps.
This lets us verify that imputation only changes originally missing entries.

In [ ]:
missing_before_imputation = df_clean.isna().sum()
wealth_observed_before_imputation = df_clean["wealth_try"].dropna().copy()
requested_imputation_columns = [
    "wealth_try", "monthly_wage_try", "height_cm", "weight_kg", "home_sqm"
]
observed_requested_values = {}
for column in requested_imputation_columns:
    observed_requested_values[column] = df_clean[column].dropna().copy()

### Fill financial gaps with medians

The median reduces the influence of valid extreme financial values.
Estimate it from the retained observed records, then fill only missing wealth and wage entries.
Wealth remains in TRY; monthly wage remains in TRY per month.

In [ ]:
median_columns = ["wealth_try", "monthly_wage_try"]
median_fill_values = df_clean[median_columns].median()
assert median_fill_values.notna().all()
df_clean[median_columns] = df_clean[median_columns].fillna(median_fill_values)
display(median_fill_values.rename("median_fill_value").to_frame())

### Fill body-measurement gaps with means

Height and weight are comparatively concentrated here, so their means provide a simple baseline.
Mean imputation reduces variation at the filled positions and does not represent uncertainty.

In [ ]:
mean_columns = ["height_cm", "weight_kg"]
mean_fill_values = df_clean[mean_columns].mean()
assert mean_fill_values.notna().all()
df_clean[mean_columns] = df_clean[mean_columns].fillna(mean_fill_values)
display(mean_fill_values.rename("mean_fill_value").to_frame())

In [ ]:
simple_fill_values = pd.concat([median_fill_values, mean_fill_values])
imputation_report = pd.DataFrame(index=simple_fill_values.index)
imputation_report["method"] = ["median", "median", "mean", "mean"]
imputation_report["observed_donors"] = len(df_clean) - missing_before_imputation
imputation_report["filled_count"] = missing_before_imputation
imputation_report["constant_fill_value"] = simple_fill_values
imputation_report.index.name = "column"
display(imputation_report.round(3))

### Prepare predictors for five-neighbour home-area imputation

Use age, height, weight, education, wage and `log1p(wealth)` to find similar people.
The logarithm compresses the wealth tail; standardization makes predictor scales comparable.
Earlier fills provide complete financial and body predictors. Keep the missing-home mask.

In [ ]:
home_missing_mask = df_clean["home_sqm"].isna()
assert df_clean["wealth_try"].ge(0).all()
knn_predictors = ["age", "height_cm", "weight_kg", "education_years", "monthly_wage_try"]
knn_features = df_clean[knn_predictors].copy()
knn_features["log1p_wealth_try"] = np.log1p(df_clean["wealth_try"])
knn_features["home_sqm"] = df_clean["home_sqm"]
assert knn_features.drop(columns="home_sqm").notna().all().all()
assert knn_features["home_sqm"].notna().sum() >= 5

Scale each feature using its observed centre and spread, preserving missing home areas.
For a missing target, home area contributes no coordinate to the neighbour distance.
Average five neighbours in standardized space, then restore the original measurement units.

In [ ]:
knn_scaler = StandardScaler()
standardized_features = knn_scaler.fit_transform(knn_features)
home_imputer = KNNImputer(n_neighbors=5, weights="uniform", metric="nan_euclidean")
standardized_imputed = home_imputer.fit_transform(standardized_features)
restored_features = pd.DataFrame(
    knn_scaler.inverse_transform(standardized_imputed),
    index=knn_features.index, columns=knn_features.columns,
)

Copy the estimates into **only the originally missing `home_sqm` positions**.
Five-neighbour estimates depend on the predictors and synthetic sample; they are not measurements.

In [ ]:
df_clean.loc[home_missing_mask, "home_sqm"] = restored_features.loc[home_missing_mask, "home_sqm"]
imputation_report.loc["home_sqm", "method"] = "KNN, k=5, standardized predictors including log1p wealth"
imputation_report.loc["home_sqm", "observed_donors"] = int((~home_missing_mask).sum())
imputation_report.loc["home_sqm", "filled_count"] = int(home_missing_mask.sum())
imputation_report.loc["home_sqm", "constant_fill_value"] = np.nan
imputation_records = imputation_report.reset_index().to_dict("records")
print("Home areas imputed:", int(home_missing_mask.sum()))
display(df_clean.loc[home_missing_mask, ["person_id", "home_sqm"]].round(2))

### Verify what changed

Every observed requested value must match its pre-imputation snapshot.
All requested numeric gaps should be filled; `df` and `df_raw` must still match exactly.

In [ ]:
for column, observed in observed_requested_values.items():
    pd.testing.assert_series_equal(df_clean.loc[observed.index, column], observed)
assert df_clean[requested_imputation_columns].notna().all().all()
assert df_clean["home_sqm"].gt(0).all()
pd.testing.assert_frame_equal(df, df_raw)
print("Requested gaps filled; observed values and the Python-generated source are unchanged.")

### Compare wealth on raw and logarithmic scales

Use the **same observed, nonnegative values** in both views, before any imputed wealth is added.
`log1p` supports zero and compresses the right tail; compare the measured skewness below.
The transformation changes interpretation, does not prove normality, and retains valid wealthy people.

In [ ]:
wealth_for_transform = wealth_observed_before_imputation
assert wealth_for_transform.ge(0).all()
wealth_log1p = np.log1p(wealth_for_transform)
wealth_skewness = pd.DataFrame(index=["Observed wealth, TRY", "log1p(observed wealth, TRY)"])
wealth_skewness["observation_count"] = [len(wealth_for_transform), len(wealth_log1p)]
wealth_skewness["sample_skewness"] = [wealth_for_transform.skew(), wealth_log1p.skew()]
wealth_skewness.index.name = "scale"
display(wealth_skewness.round(3))

In [ ]:
wealth_figure = Figure(figsize=(12, 4))
wealth_axes = wealth_figure.subplots(1, 2)
wealth_axes[0].hist(wealth_for_transform, bins=35, color="#2171b5", edgecolor="white")
wealth_axes[0].set(title="Observed wealth: raw scale", xlabel="Wealth (TRY)", ylabel="Persons")
wealth_axes[0].ticklabel_format(axis="x", style="sci", scilimits=(0, 0))

In [ ]:
wealth_axes[1].hist(wealth_log1p, bins=35, color="#238b45", edgecolor="white")
wealth_axes[1].set(title="Same observations: log1p scale", xlabel="log(1 + wealth in TRY)", ylabel="Persons")
wealth_figure.suptitle("Wealth transformation: valid tails are retained")
wealth_figure.tight_layout()
display(wealth_figure)

### Review missingness before and after cleaning

Compare the raw sample, the retained records before imputation, and the final cleaned sample.
Remaining missing fields are reported explicitly. Other fields are preserved.
`df_clean` and the reports remain in Python memory for further analysis.

In [ ]:
quality_summary = missing_analysis.rename(columns={
    "nonmissing_count": "raw_nonmissing_count",
    "missing_count": "raw_missing_count", "missing_pct": "raw_missing_pct",
}).copy()
quality_summary["missing_after_record_exclusion"] = missing_before_imputation
quality_summary["missing_after_imputation"] = df_clean.isna().sum()
quality_summary["clean_missing_pct"] = (100 * df_clean.isna().mean()).round(2)
quality_summary["values_imputed"] = missing_before_imputation - df_clean.isna().sum()

In [ ]:
quality_summary["imputation_method"] = "not requested; preserved"
for column, method in imputation_report["method"].items():
    quality_summary.loc[column, "imputation_method"] = method
quality_summary["raw_more_than_75pct_identical"] = concentration_analysis["more_than_75pct_identical"]
display(quality_summary)
remaining_missing = df_clean.isna().sum()
print("Remaining missing fields:")
display(remaining_missing[remaining_missing.gt(0)].rename("missing_count").to_frame())

### Recalculate descriptive IQR flags after preparation

Re-estimate quartiles on `df_clean` and count remaining flags.
These flags remain in the data: the only excluded row had an impossible schooling measurement.

In [ ]:
clean_numeric = df_clean[numeric_cols]
clean_iqr_summary = pd.DataFrame(index=numeric_cols)
clean_iqr_summary["observed_count"] = clean_numeric.notna().sum()
clean_iqr_summary["q1"] = clean_numeric.quantile(0.25)
clean_iqr_summary["q3"] = clean_numeric.quantile(0.75)
clean_iqr_summary["iqr"] = clean_iqr_summary["q3"] - clean_iqr_summary["q1"]
clean_iqr_summary["lower_fence"] = clean_iqr_summary["q1"] - 1.5 * clean_iqr_summary["iqr"]
clean_iqr_summary["upper_fence"] = clean_iqr_summary["q3"] + 1.5 * clean_iqr_summary["iqr"]

In [ ]:
clean_below = clean_numeric.lt(clean_iqr_summary["lower_fence"], axis="columns")
clean_above = clean_numeric.gt(clean_iqr_summary["upper_fence"], axis="columns")
clean_iqr_flags = clean_below | clean_above
clean_iqr_summary["below_fence_count"] = clean_below.sum()
clean_iqr_summary["above_fence_count"] = clean_above.sum()
clean_iqr_summary["iqr_flag_count"] = clean_iqr_flags.sum()
display(clean_iqr_summary.round(2))
print("Cleaned records with an IQR flag (retained):", int(clean_iqr_flags.any(axis=1).sum()))

In [ ]:
preprocessing_metadata = {
    "data_kind": "synthetic educational exercise; not real respondents",
    "raw_rows": len(df_raw), "clean_rows": len(df_clean),
    "excluded_measurement_error_rows": len(measurement_error_rows),
    "measurement_error_rule": "education_years < 0 or education_years > 100",
    "knn_neighbors": 5, "knn_predictors": knn_features.columns.drop("home_sqm").tolist(),
    "imputations": imputation_records,
}

In [ ]:
preprocessing_metadata["raw_source_unchanged"] = df.equals(df_raw)
preprocessing_metadata["observed_requested_values_preserved"] = True
preprocessing_metadata["remaining_missing"] = remaining_missing[remaining_missing.gt(0)].to_dict()
print(f"Ready for further analysis: df_clean has {df_clean.shape[0]} rows and {df_clean.shape[1]} columns.")
display(df_clean.head())

## Discussion and next practice
1. Why is 1,000 years of education an error, while a large wealth value may be valid?
2. How do the mean and median respond to a long upper tail?
3. Why standardize features before finding nearest neighbours?
4. What additional evidence would be needed for a causal interpretation?
5. How might income-related missingness change the choice of imputation method?

Restart the runtime and run all cells before sharing your work.

The raw data remain in `df_raw`, the cleaned data in `df_clean`, and the quality tables in notebook variables. The lesson generates and analyzes everything in memory.

References: [pandas](https://pandas.pydata.org/docs/), [seaborn](https://seaborn.pydata.org/), [scikit-learn imputation](https://scikit-learn.org/stable/modules/impute.html).